# 04 — Policy review priority score

Build a **transparent, explainable** first version of the
`policy_review_priority_score`: a 0-100 triage signal that flags a geography ×
period as showing elevated labour-market stress that warrants **human review**.

> **This is NOT a social-assistance need score.** It is not an eligibility, benefit, approval, denial, or reduction decision. It only helps a human analyst decide *where to look first*. Every score carries its drivers (`score_explanation`) and a `confidence_flag`; missing inputs lower confidence and are never imputed.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # import project modules
import utils
print("Project root:", utils.ROOT)
print("Raw data:    ", utils.RAW)
print("Processed:   ", utils.PROCESSED)
print("Outputs:     ", utils.OUTPUTS)
utils.ensure_dirs()

In [ ]:
import pandas as pd, numpy as np
import data_cleaning as dc, scoring as sc
from datetime import date

## Inputs and weights (editable — the AI Council reviews these)

In [ ]:
print("Weights:"); [print(f"  {k:<18} {v}") for k,v in sc.WEIGHTS.items()]
print("\nNormalisation anchors (value@0 -> value@1):")
[print(f"  {k:<18} {v}") for k,v in sc.ANCHORS.items()]

Each component is normalised to 0-1 against **fixed, documented anchors**, then combined as `score = 100 × Σ(weight·component) / Σ(weight of *available* components)`. Declines in employment/participation are treated as stress (sign-flipped). A missing component is dropped from that row's average and lowers `score_confidence` — it never defaults to zero.

## Build the headline panel and score it

In [ ]:
clean = pd.read_csv(utils.PROCESSED / "labour_force_clean.csv", low_memory=False)
panel = dc.lfs_headline_features(clean)

# Context columns (income/affordability/demographics) are NaN until those tables
# are downloaded and merged in notebook 05 — flagged, never invented.
for col in ["low_income_rate","housing_pressure_proxy","income_value","population"]:
    if col not in panel.columns: panel[col] = np.nan

scored = sc.add_scores(panel)
print(f"scored {len(scored):,} rows")
scored.sort_values("ref_date").groupby("geo").tail(1)[
    ["geo","ref_date","unemployment_rate","youth_unemployment_rate",
     "policy_review_priority_score","confidence_flag"]
].sort_values("policy_review_priority_score", ascending=False)

## Inspect one explanation

In [ ]:
row = scored.sort_values("policy_review_priority_score", ascending=False).iloc[0]
print(row["geo"], row["ref_date"])
print(row["score_explanation"])

## Save score output

In [ ]:
scored.to_csv(utils.PROCESSED / "policy_triage_panel.csv", index=False)
print("wrote data/processed/policy_triage_panel.csv")

### Limitations to surface on the dashboard

With only the LFS downloaded, confidence is capped at **medium (75%)** because the low-income and housing-pressure components are missing. Download the income / affordability / demographics tables (notebooks 01-02) to raise confidence. The housing field is always a **proxy** (shelter-CPI growth), not a measured affordability rate. The AI Council reviews the weights and any policy reading before deployment.